# 01 · Activation Functions and Analytical vs. Numerical Derivatives

In neural networks, **activation functions** introduce the non-linear mappings required for deep architectures to approximate arbitrary continuous functions (Universal Approximation Theorem). When training networks via backpropagation, computing the derivative of each activation function is essential to propagate error gradients from output layers back to input weights.

In this notebook, we explore:
1. The mathematical definition and analytical derivative of **Sigmoid**, **Tanh**, and **Swish**.
2. Step-by-step algebraic derivations of their derivative identities.
3. Numerical differentiation using finite difference schemes (single-sided vs. central difference).
4. Vectorized derivative computation using NumPy's `np.gradient`.
5. Visualizing activation functions alongside their derivatives and analyzing **vanishing gradient** saturation regions.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np


def plot_function_and_derivative(x, f_x, df_x, title="Function and Derivative"):
    """Utility function to display function and its derivative with centered axes."""
    _fig, ax = plt.subplots(figsize=(10, 5))
    ax.spines["left"].set_position("center")
    ax.spines["bottom"].set_position("zero")
    ax.spines["right"].set_color("none")
    ax.spines["top"].set_color("none")
    ax.xaxis.set_ticks_position("bottom")
    ax.yaxis.set_ticks_position("left")

    ax.plot(x, f_x, color="#d35400", linewidth=2.5, label="Function f(x)")
    ax.plot(x, df_x, color="#1abd15", linewidth=2.5, linestyle="--", label="Derivative f'(x)")

    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper left", frameon=True)
    ax.set_title(title, pad=20)
    plt.tight_layout()
    plt.show()

## 1. Sigmoid Function and its Derivative

The standard logistic Sigmoid function maps any real-valued number into the open interval $(0, 1)$:

$$\sigma(x) = \frac{1}{1 + e^{-x}} = (1 + e^{-x})^{-1}$$

### Analytical Derivation
Applying the chain rule:

$$\sigma'(x) = -1 \cdot (1 + e^{-x})^{-2} \cdot \frac{d}{dx}(1 + e^{-x}) = - (1 + e^{-x})^{-2} \cdot (-e^{-x}) = \frac{e^{-x}}{(1 + e^{-x})^2}$$

We can re-express the numerator by adding and subtracting $1$:

$$\frac{e^{-x}}{(1 + e^{-x})^2} = \frac{(1 + e^{-x}) - 1}{(1 + e^{-x})^2} = \frac{1}{1 + e^{-x}} - \frac{1}{(1 + e^{-x})^2}$$

Factoring out $\frac{1}{1 + e^{-x}}$:

$$\sigma'(x) = \frac{1}{1 + e^{-x}} \left(1 - \frac{1}{1 + e^{-x}}\right) = \sigma(x)(1 - \sigma(x))$$

This identity means that during backpropagation, $\sigma'(x)$ can be computed directly from the forward pass output $\sigma(x)$ with a single subtraction and multiplication, without re-evaluating the exponential function.


In [ ]:
def sigmoid(x: float) -> float:
    """Compute standard logistic sigmoid function."""
    return 1.0 / (1.0 + math.exp(-x))


def d_sigmoid(x: float) -> float:
    """Compute exact analytical derivative of sigmoid."""
    s = sigmoid(x)
    return s * (1.0 - s)


# Verify analytical properties
assert math.isclose(sigmoid(0.0), 0.5, rel_tol=1e-7)
assert math.isclose(d_sigmoid(0.0), 0.25, rel_tol=1e-7)
print(f"sigmoid(0.0) = {sigmoid(0.0):.4f}")
print(f"d_sigmoid(0.0) = {d_sigmoid(0.0):.4f} (Maximum gradient)")

In [ ]:
# Generate grid over [-10, 10]
x_data = [x / 10.0 for x in range(-100, 101)]
y_data = [sigmoid(x) for x in x_data]
dy_data = [d_sigmoid(x) for x in x_data]

plot_function_and_derivative(
    x_data, y_data, dy_data, title="Sigmoid: f(x) and Analytical Derivative f'(x)"
)

## 2. Numerical Differentiation (Finite Difference Approximations)

When analytical forms are unavailable or to verify gradient implementations (Gradient Checking), numerical differentiation approximates the limit definition:

$$f'(x) \approx \frac{f(x + \epsilon) - f(x)}{\epsilon} \quad \text{(Single-sided forward difference)}$$
$$f'(x) \approx \frac{f(x + \epsilon/2) - f(x - \epsilon/2)}{\epsilon} \quad \text{(Symmetric central difference)}$$

Central difference offers second-order accuracy $\mathcal{O}(\epsilon^2)$, whereas forward difference has first-order error $\mathcal{O}(\epsilon)$.


In [ ]:
def autograd_forward(f, x: float, epsilon: float = 1e-6) -> float:
    """Single-sided forward difference approximation."""
    return (f(x + epsilon) - f(x)) / epsilon


def autograd_central(f, x: float, epsilon: float = 1e-6) -> float:
    """Symmetric central difference approximation."""
    return (f(x + epsilon / 2.0) - f(x - epsilon / 2.0)) / epsilon


# Error check at x = 1.5
x_test = 1.5
exact = d_sigmoid(x_test)
num_fwd = autograd_forward(sigmoid, x_test)
num_cnt = autograd_central(sigmoid, x_test)

print(f"At x = {x_test}:")
print(f"  Exact derivative:       {exact:.10f}")
print(f"  Forward difference:     {num_fwd:.10f} (Error: {abs(exact - num_fwd):.2e})")
print(f"  Central difference:     {num_cnt:.10f} (Error: {abs(exact - num_cnt):.2e})")

assert abs(exact - num_cnt) < 1e-10

In [ ]:
# Vectorized differentiation via np.gradient
x_np = np.linspace(-10.0, 10.0, 201)
y_np = 1.0 / (1.0 + np.exp(-x_np))
dy_np = np.gradient(y_np, x_np)
dy_exact = y_np * (1.0 - y_np)

max_error = np.max(np.abs(dy_np - dy_exact))
print(f"Max absolute discrepancy between np.gradient and analytical derivative: {max_error:.2e}")
assert max_error < 1e-3

plot_function_and_derivative(
    x_np, y_np, dy_np, title="Sigmoid: Numerical Derivative with np.gradient"
)

## 3. Hyperbolic Tangent (Tanh) Function and its Derivative

The hyperbolic tangent function is zero-centered and ranges across $(-1, 1)$:

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} = 1 - \frac{2}{e^{2x} + 1} = \frac{2}{1 + e^{-2x}} - 1$$

### Analytical Derivation
Using the quotient rule $\left(\frac{u}{v}\right)' = \frac{u'v - uv'}{v^2}$ with $u = e^x - e^{-x}$ and $v = e^x + e^{-x}$:

$$\tanh'(x) = \frac{(e^x + e^{-x})(e^x + e^{-x}) - (e^x - e^{-x})(e^x - e^{-x})}{(e^x + e^{-x})^2}$$
$$\tanh'(x) = \frac{(e^x + e^{-x})^2 - (e^x - e^{-x})^2}{(e^x + e^{-x})^2} = 1 - \left(\frac{e^x - e^{-x}}{e^x + e^{-x}}\right)^2 = 1 - \tanh^2(x)$$

Alternatively, using the exponential form $\tanh(x) = 2(1 + e^{-2x})^{-1} - 1$:

$$\tanh'(x) = 2 \cdot (-1)(1 + e^{-2x})^{-2} \cdot (-2e^{-2x}) = \frac{4e^{-2x}}{(1 + e^{-2x})^2} = 1 - \tanh^2(x)$$


In [ ]:
def tanh(x: float) -> float:
    """Compute hyperbolic tangent function."""
    return (math.exp(x) - math.exp(-x)) / (math.exp(x) + math.exp(-x))


def d_tanh(x: float) -> float:
    """Compute exact analytical derivative of tanh."""
    t = tanh(x)
    return 1.0 - t * t


# Verify properties
assert math.isclose(tanh(0.0), 0.0, abs_tol=1e-7)
assert math.isclose(d_tanh(0.0), 1.0, abs_tol=1e-7)
print(f"tanh(0.0) = {tanh(0.0):.4f}")
print(f"d_tanh(0.0) = {d_tanh(0.0):.4f} (Zero-centered, unit peak gradient)")

y_tanh = [tanh(x) for x in x_data]
dy_tanh = [d_tanh(x) for x in x_data]
plot_function_and_derivative(
    x_data, y_tanh, dy_tanh, title="Tanh: f(x) and Analytical Derivative f'(x)"
)

## 4. Swish Activation Function

Introduced by Ramachandran et al. (2017), the **Swish** activation function is defined as:

$$\text{swish}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

### Analytical Derivation
Applying the product rule $(u \cdot v)' = u'v + uv'$ where $u = x$ and $v = \sigma(x)$:

$$\text{swish}'(x) = (x)' \cdot \sigma(x) + x \cdot \sigma'(x) = 1 \cdot \sigma(x) + x \cdot \sigma(x)(1 - \sigma(x))$$

Since $x \cdot \sigma(x) = \text{swish}(x)$:

$$\text{swish}'(x) = \sigma(x) + \text{swish}(x)(1 - \sigma(x)) = \text{swish}(x) + \sigma(x)(1 - \text{swish}(x))$$

Swish is smooth, non-monotonic, and allows a small gradient to propagate when $x < 0$, avoiding the "dead neuron" issue of hard ReLUs while outperforming ReLU across many deep architectures.


In [ ]:
def swish(x: float) -> float:
    """Compute Swish activation function."""
    return x * sigmoid(x)


def d_swish(x: float) -> float:
    """Compute exact derivative of Swish."""
    s = sigmoid(x)
    sw = x * s
    return sw + s * (1.0 - sw)


# Test numerical agreement
x_val = -1.2
exact_swish_d = d_swish(x_val)
num_swish_d = autograd_central(swish, x_val)
assert abs(exact_swish_d - num_swish_d) < 1e-8
print(f"Swish derivative at x = {x_val}: {exact_swish_d:.6f} (Numerical: {num_swish_d:.6f})")

y_swish = [swish(x) for x in x_data]
dy_swish = [d_swish(x) for x in x_data]
plot_function_and_derivative(
    x_data, y_swish, dy_swish, title="Swish: f(x) and Analytical Derivative f'(x)"
)

## 5. Derivative Comparison and the Vanishing Gradient Problem

Let us plot the derivative curves of Sigmoid, Tanh, and Swish on a single figure to observe the gradient magnitude and vanishing gradient dynamics.


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(x_data, dy_data, label="Sigmoid Derivative (max = 0.25)", color="#e74c3c", linewidth=2.5)
plt.plot(x_data, dy_tanh, label="Tanh Derivative (max = 1.00)", color="#2980b9", linewidth=2.5)
plt.plot(x_data, dy_swish, label="Swish Derivative (max ≈ 1.09)", color="#27ae60", linewidth=2.5)

plt.axhline(0, color="black", linestyle="--", alpha=0.5)
plt.axvline(0, color="black", linestyle="--", alpha=0.5)
plt.xlabel("Input x", fontsize=12)
plt.ylabel("Gradient df/dx", fontsize=12)
plt.title(
    "Comparison of Activation Derivatives: Vanishing Gradients in Saturated Regimes", fontsize=14
)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Key numerical takeaways
print("Key Gradient Insights:")
print(
    "1. Sigmoid max derivative is 0.25 at x=0; for |x| > 4, d_sigmoid < 0.018 (rapid gradient death)."
)
print("2. Tanh is 4x steeper at origin (peak 1.0), but still saturates to zero for |x| > 3.")
print(
    "3. Swish remains non-zero for negative inputs and asymptotically approaches 1 for large positive inputs."
)